#### Import modules

In [1]:
# Simple import setup
import sys
import os
sys.path.append('..')  # Add parent directory to path

# Import all necessary modules
import pandas as pd
import math
import matplotlib.pyplot as plt

# Import from the new modular structure
from src_code import (
    pf_enviroment, initialize_powerfactory, activate_project, list_and_select_study_case,
    list_and_activate_operation_scenario, run_contingency_analysis, process_cargabilidad, optimize_generators_for_substations,
    create_static_generator, calculate_power_limits, delete_generator, update_generator_power, cleanup_existing_generator, cleanup_all_test_generators
    )

### PowerFactory environment definition ------------

#### Initialize PowerFactory

In [2]:
# Añadir la ruta del entorno de PowerFactory
dig_path = r'C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9'
pf_enviroment(dig_path)

PowerFactory environment initialized with path: C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9


#### Get PowerFactory application

In [3]:
# Initialize PowerFactory application
app = initialize_powerfactory()

PowerFactory application connected successfully!


#### Activate project

In [4]:
project_name = "39 Bus New England System"
project = activate_project(app, project_name)

Proyecto '39 Bus New England System' activado con exito.


#### Activate study case

In [5]:
#### Clean up existing test generators


In [6]:
# Clean up any existing test generators before running new tests
cleanup_all_test_generators(app)


Limpiando todos los generadores de prueba existentes...
Encontrado generador de prueba: 'Gen_Estatico_Bus 03'
Eliminando generador: 'Gen_Estatico_Bus 03'
Encontrado cubículo de prueba: 'Cubicle_Gen_Bus 03'
Eliminando cubículo: 'Cubicle_Gen_Bus 03'
Limpieza completada. Eliminados 1 generadores y 1 cubículos.


In [7]:
# Activate the desired study case
study_case_name = "1. Power Flow"  # Replace with your specific case name
active_study_case = list_and_select_study_case(app, study_case_name)

List of study cases:
2.1 Simulation Fault Bus 16 Stable
2.2 Simulation Fault Bus 16 Unstable
2.3 Simulation Fault Bus 31 Stable
2.4 Simulation Fault Bus 31 Unstable
2.5 Simulation Fault Line 2-3 Stable
2.6 Simulation Fault Line 2-3 Unstable
3. Small Signal Analysis (Eigenvalues)
4. EMT Simulation Fault Bus 03
1. Power Flow
Study case '1. Power Flow' activated.


#### Activate operation scenario

In [8]:
operation_scenario_name = "Basic Load Flow"
active_scenario = list_and_activate_operation_scenario(app, operation_scenario_name)

List of operation scenarios:
EMT
Basic Load Flow
Operation scenario 'Basic Load Flow' activated.


### Run contingency analysis ---------------

#### Get network data

In [9]:
project = app.GetActiveProject()
network_data = project.GetContents('Network Model.IntPrjfolder\\Network Data', 1)[0]
hoja = 'Grid'

#### Get buses

In [10]:
# buses = app.GetCalcRelevantObjects('*.ElmTerm')
# buses = [bus.loc_name for bus in buses]

#### Run search for max contingency

In [11]:
# initial_potencia = 1
# factor_potencia = 0.95
# max_cargabilidad = 110
# threshold_inconvergence = 10
# df_results = optimize_generators_for_substations(
#     app=app,  
#     substations=buses,  # Should be a list of strings, not PowerFactory objects
#     network_data=network_data, 
#     hoja=hoja, 
#     initial_potencia=initial_potencia, 
#     factor_potencia=factor_potencia, 
#     max_cargabilidad=max_cargabilidad, 
#     threshold_inconvergence=threshold_inconvergence
# )

#### Test the corrected reactive power limits


In [ ]:
# Complete Contingency Analysis Optimizer
import numpy as np

def run_contingency_analysis(app):
    """Run contingency analysis and return results"""
    print("Iniciando análisis de contingencia N-1.")
    app.ClearOutputWindow()
    
    # Get the contingency analysis module
    contingency_analysis = app.GetFromStudyCase('*.ComSimoutage')
    if contingency_analysis is None:
        print("Error: No se encontró el módulo de análisis de contingencia")
        return None
    
    contingency_analysis.iopt_Linear = 0
    contingency_analysis.loadmax = 50
    contingency_analysis.vlmin = 0.9
    contingency_analysis.vlmax = 1.1
    contingency_analysis.vmax_step = 5

    contingency_analysis.Execute()
    
    elmres = app.GetFromStudyCase('Contingency Analysis AC.ElmRes')
    comres = app.GetFromStudyCase('ComRes')
    comres.iopt_exp = 6
    comres.iopt_csel = 0
    comres.pResult = elmres
    comres.f_name = r'Resultados.csv'
    comres.Execute()

    print("Cargando resultados del archivo 'Resultados.csv'.")
    df = pd.read_csv('Resultados.csv', encoding='latin1', low_memory=False)
    return df

def process_cargabilidad(df):
    """Process loading results from contingency analysis CSV"""
    print("Procesando resultados de cargabilidad desde Resultados.csv...")
    
    # Get the header row (row 0) and parameter row (row 1)
    headers = df.iloc[0].values
    parameters = df.iloc[1].values
    
    # Find columns that contain "Max. loading in %" parameter
    max_loading_indices = []
    line_names = []
    
    for i, param in enumerate(parameters):
        if param == "Max. loading in %":
            max_loading_indices.append(i)
            line_names.append(headers[i])
    
    print(f"Encontradas {len(max_loading_indices)} líneas con datos de cargabilidad máxima.")
    
    # Extract maximum loading data for each line
    line_load_data = []
    
    for i, (idx, line_name) in enumerate(zip(max_loading_indices, line_names)):
        # Get all data for this line (skip first 2 rows which are headers)
        line_data = df.iloc[2:, idx].values
        
        # Convert to numeric, handling '----' as NaN
        numeric_data = []
        for value in line_data:
            if value == '   ----' or value == '----' or pd.isna(value):
                numeric_data.append(np.nan)
            else:
                try:
                    numeric_data.append(float(str(value).strip()))
                except:
                    numeric_data.append(np.nan)
        
        # Find the maximum loading for this line
        valid_data = [x for x in numeric_data if not np.isnan(x)]
        if valid_data:
            max_loading = max(valid_data)
            line_load_data.append({
                'Linea': line_name,
                'Cargabilidad_Maxima': max_loading,
                'Datos_Disponibles': len(valid_data)
            })
        else:
            print(f"⚠️  Línea '{line_name}' no tiene datos válidos (todos son '----')")
    
    # Create DataFrame and sort by maximum loading
    line_load_df = pd.DataFrame(line_load_data)
    
    if not line_load_df.empty:
        # Filter out lines with very low loading (likely disconnected)
        line_load_df = line_load_df[line_load_df['Cargabilidad_Maxima'] > 1.0]
        
        # Sort by maximum loading
        line_load_df = line_load_df.sort_values(by='Cargabilidad_Maxima', ascending=False).reset_index(drop=True)
        
        print(f"Procesamiento completado. {len(line_load_df)} líneas con cargabilidad > 1%")
        print(f"Top 5 líneas más cargadas:")
        for i, row in line_load_df.head().iterrows():
            print(f"  {i+1}. {row['Linea']}: {row['Cargabilidad_Maxima']:.2f}%")
    else:
        print("⚠️  No se encontraron datos válidos de cargabilidad.")
    
    return line_load_df

def show_iteration_details(df, iteration_num, substation, current_potencia):
    """Show detailed iteration results from CSV data"""
    print(f"\n📊 DETALLES DE ITERACIÓN {iteration_num} - {substation} ({current_potencia} MW)")
    print("="*60)
    
    # Get the header row (row 0) and parameter row (row 1)
    headers = df.iloc[0].values
    parameters = df.iloc[1].values
    
    # Find columns that contain "Max. loading in %" parameter
    max_loading_indices = []
    line_names = []
    
    for i, param in enumerate(parameters):
        if param == "Max. loading in %":
            max_loading_indices.append(i)
            line_names.append(headers[i])
    
    # Show top 10 most loaded lines
    line_loads = []
    for i, (idx, line_name) in enumerate(zip(max_loading_indices, line_names)):
        # Get all data for this line (skip first 2 rows which are headers)
        line_data = df.iloc[2:, idx].values
        
        # Convert to numeric, handling '----' as NaN
        numeric_data = []
        for value in line_data:
            if value == '   ----' or value == '----' or pd.isna(value):
                numeric_data.append(np.nan)
            else:
                try:
                    numeric_data.append(float(str(value).strip()))
                except:
                    numeric_data.append(np.nan)
        
        # Find the maximum loading for this line
        valid_data = [x for x in numeric_data if not np.isnan(x)]
        if valid_data:
            max_loading = max(valid_data)
            line_loads.append((line_name, max_loading))
    
    # Sort by loading and show top 10
    line_loads.sort(key=lambda x: x[1], reverse=True)
    
    print("Top 10 líneas más cargadas:")
    for i, (line_name, loading) in enumerate(line_loads[:10]):
        status = "🔴 CRÍTICA" if loading >= 100 else "🟡 ALTA" if loading >= 80 else "🟢 NORMAL"
        print(f"  {i+1:2d}. {line_name:15s}: {loading:6.2f}% {status}")
    
    if len(line_loads) > 10:
        print(f"  ... y {len(line_loads) - 10} líneas más")
    
    # Show lines approaching 110% limit
    critical_lines = [line for line, loading in line_loads if loading >= 105]
    if critical_lines:
        print(f"\n⚠️  Líneas cercanas al límite (≥105%): {len(critical_lines)}")
        for line_name, loading in critical_lines[:5]:
            print(f"    - {line_name}: {loading:.2f}%")
    
    return line_loads

def optimize_generators_for_substations(app, network_data, hoja, substations, initial_potencia=1, factor_potencia=0.95, max_cargabilidad=110, threshold_inconvergence=10):
    """
    Optimize generators for each substation using contingency analysis
    """
    results = []  # Lista para almacenar los resultados

    for substation in substations:
        current_potencia = initial_potencia
        print(f"\n{'='*60}")
        print(f"Optimizando generador para la subestación '{substation}'.")
        print(f"{'='*60}")

        # Crear generador inicial
        bus_voltage, p_gen, q_gen, static_generator, cubicle = create_static_generator(
            app, network_data, hoja, substation, current_potencia, factor_potencia
        )
        
        if static_generator is None:
            print(f"❌ Error: No se pudo crear el generador en la subestación '{substation}'.")
            continue

        last_max_line_load = None
        iteration = 0
        max_iterations = 50  # Safety limit

        while iteration < max_iterations:
            iteration += 1
            print(f"\n--- Iteración {iteration} para {substation} ---")
            
            # 1. Ejecutar flujo de potencia para verificar voltajes y potencias
            print("Ejecutando flujo de potencia para verificar sistema...")
            power_flow = app.GetFromStudyCase('ComLdf')
            power_flow.Execute()
            
            # Obtener información actual del generador y barra DESPUÉS del load flow
            current_p_gen = static_generator.GetAttribute('c:p')
            current_q_gen = static_generator.GetAttribute('c:q')
            
            # Get voltage from the bus terminal, not the generator
            bus_terminal = static_generator.term
            current_bus_voltage = bus_terminal.GetAttribute('m:u')
            
            print(f"📊 Estado actual: P = {current_p_gen:.2f} MW, Q = {current_q_gen:.2f} MVar, V = {current_bus_voltage:.4f} pu")
            
            # Verificar límites de voltaje (0.9 - 1.1 pu)
            if current_bus_voltage < 0.9 or current_bus_voltage > 1.1:
                print(f"⚠️  Voltaje fuera de límites: {current_bus_voltage:.4f} pu (límites: 0.9 - 1.1 pu)")
                print(f"🎯 Límite de voltaje alcanzado: Potencia máxima segura = {current_potencia - 1} MW")
                
                # Guardar el resultado
                results.append({
                    'Subestacion': substation,
                    'Potencia_Maxima': current_potencia - 1,
                    'Linea_Critica': 'Voltage Limit',
                    'Cargabilidad_Maxima': f'V={current_bus_voltage:.4f}',
                    'Iteraciones': iteration - 1,
                    'P_Final': current_p_gen,
                    'Q_Final': current_q_gen
                })
                
                # Eliminar el generador estático y el cubículo
                print(f"Limpiando generador en {substation}...")
                delete_generator(static_generator, cubicle)
                break
            
            # 2. Ejecutar el análisis de contingencia N-1 y obtener los resultados
            print("Ejecutando análisis de contingencia...")
            df = run_contingency_analysis(app)
            if df is None:
                print("❌ Error en análisis de contingencia. Deteniendo optimización.")
                break
                
            # Show detailed iteration results
            line_loads = show_iteration_details(df, iteration, substation, current_potencia)
            
            # Process the data for optimization logic
            line_load_df = process_cargabilidad(df)

            # Verificar si alguna línea excede el límite de cargabilidad
            if not line_load_df.empty:
                max_line_load = line_load_df['Cargabilidad_Maxima'].max()
                max_line = line_load_df[line_load_df['Cargabilidad_Maxima'] == max_line_load]['Linea'].values[0]
            else:
                print("⚠️  No hay datos de cargabilidad válidos")
                max_line_load = 0
                max_line = "N/A"

            print(f"\n🎯 RESUMEN: Potencia actual = {current_potencia} MW")
            print(f"Max cargabilidad = {max_line_load:.2f}%, Línea crítica = {max_line}")

            # Detectar inconvergencia si el incremento supera el umbral definido
            if last_max_line_load is not None and (max_line_load - last_max_line_load) > threshold_inconvergence and max_line_load > max_cargabilidad:
                print(f"⚠️  Advertencia: Inconvergencia detectada. La cargabilidad saltó {max_line_load - last_max_line_load:.2f}%")
                print(f"Aumentando potencia y volviendo a intentar...")
                # Aumentar la potencia en 1 MW y continuar
                current_potencia += 1
                update_generator_power(static_generator, current_potencia, factor_potencia)
                continue

            last_max_line_load = max_line_load

            # Verificar si se alcanzó el límite de cargabilidad (110%)
            if max_line_load >= max_cargabilidad:
                print(f"🎯 Límite de cargabilidad alcanzado: Potencia máxima segura = {current_potencia} MW")
                print(f"Línea crítica = {max_line} (Cargabilidad: {max_line_load:.2f}%)")

                # Guardar el resultado
                results.append({
                    'Subestacion': substation,
                    'Potencia_Maxima': current_potencia,
                    'Linea_Critica': max_line,
                    'Cargabilidad_Maxima': max_line_load,
                    'Iteraciones': iteration,
                    'P_Final': current_p_gen,
                    'Q_Final': current_q_gen
                })

                # Eliminar el generador estático y el cubículo
                print(f"Limpiando generador en {substation}...")
                delete_generator(static_generator, cubicle)
                break

            # Aumentar la potencia del generador estático existente
            current_potencia += 1
            print(f"🔄 Aumentando potencia a {current_potencia} MW...")
            update_generator_power(static_generator, current_potencia, factor_potencia)

        if iteration >= max_iterations:
            print(f"⚠️  Máximo de iteraciones alcanzado para {substation}")
            # Clean up
            delete_generator(static_generator, cubicle)

    # Convertir los resultados a un DataFrame
    df_results = pd.DataFrame(results)
    print(f"\n{'='*60}")
    print("RESULTADOS FINALES:")
    print(f"{'='*60}")
    if not df_results.empty:
        print(df_results.to_string(index=False))
    else:
        print("No se obtuvieron resultados.")
    
    return df_results

# Main execution
print("🚀 INICIANDO ANÁLISIS DE CONTINGENCIA CON OPTIMIZACIÓN DE GENERADORES")
print("="*80)

# Clean up any existing generators first
print("Limpiando generadores existentes...")
cleanup_all_test_generators(app)

# Define substations to analyze
substations = [
    "Bus 03",
    "Bus 04", 
    "Bus 05",
    "Bus 06",
    "Bus 07",
    "Bus 08",
    "Bus 09"
]

# Parameters
initial_potencia = 1  # MW
factor_potencia = 0.95
max_cargabilidad = 110  # %
threshold_inconvergence = 10  # %

print(f"Subestaciones a analizar: {substations}")
print(f"Potencia inicial: {initial_potencia} MW")
print(f"Factor de potencia: {factor_potencia}")
print(f"Límite de cargabilidad: {max_cargabilidad}%")
print(f"Umbral de inconvergencia: {threshold_inconvergence}%")

# Run optimization
df_results = optimize_generators_for_substations(
    app=app,
    network_data=network_data,
    hoja=hoja,
    substations=substations,
    initial_potencia=initial_potencia,
    factor_potencia=factor_potencia,
    max_cargabilidad=max_cargabilidad,
    threshold_inconvergence=threshold_inconvergence
)

print(f"\n✅ ANÁLISIS COMPLETADO")
print(f"Total de subestaciones analizadas: {len(df_results)}")
if not df_results.empty:
    print(f"Potencia máxima total posible: {df_results['Potencia_Maxima'].sum()} MW")
    print(f"Promedio de potencia por subestación: {df_results['Potencia_Maxima'].mean():.2f} MW")


🚀 INICIANDO ANÁLISIS DE CONTINGENCIA CON OPTIMIZACIÓN DE GENERADORES
Limpiando generadores existentes...
Limpiando todos los generadores de prueba existentes...
Limpieza completada. Eliminados 0 generadores y 0 cubículos.
Subestaciones a analizar: ['Bus 03', 'Bus 04', 'Bus 05', 'Bus 06', 'Bus 07', 'Bus 08', 'Bus 09']
Potencia inicial: 1 MW
Factor de potencia: 0.95
Límite de cargabilidad: 110%
Umbral de inconvergencia: 10%

Optimizando generador para la subestación 'Bus 03'.
Creando generador estático en la barra 'Bus 03' con potencia activa 1 MW y factor de potencia 0.95.
Límites calculados: S = 1.05 MVA, Q_max = 0.33 MVar, Q_min = -0.33 MVar
Buscando hoja 'Grid' en 'Network Data'.
Debug: Found folder: Diagrams (type: IntPrjfolder)
Debug: Found folder: Network Data (type: IntPrjfolder)
Debug: Found folder: Operation Scenarios (type: IntPrjfolder)
Debug: Found folder: Variations (type: IntPrjfolder)
Debug: Found folder: Grid (type: IntGrfnet)
Debug: Found Grid network diagram: 'Grid' (t